# Ch10 DPO: Direct Preference Optimization 教案

**课程名称：** DPO 偏好优化：让模型学会「说人话」

**预计总时长：** 85-95 分钟

**源文件：** `Ch10_DPO/Ch10_DPO.ipynb`（共 40 个 Cell，Cell 0-39）

---

## 时间表

| 时间段 | 内容 | Cell 范围 | 时长 |
|:---|:---|:---|:---|
| 0-8 min | 开场 + 对齐动机 + RLHF vs DPO 对比 | Cell 0-3 | 8 min |
| 8-13 min | 环境准备与依赖加载 | Cell 4-6 | 5 min |
| 13-23 min | 偏好数据构造与分析 | Cell 7-8 | 10 min |
| 23-43 min | DPO 数学原理（公式推导 + 可视化） | Cell 9-14 | 20 min |
| 43-48 min | 休息 + 回顾 | — | 5 min |
| 48-58 min | 模型加载与配置 | Cell 15-18 | 10 min |
| 58-73 min | DPO 训练实战 | Cell 19-26 | 15 min |
| 73-83 min | 效果评估与隐式奖励 | Cell 27-31 | 10 min |
| 83-88 min | 生产级实现 + 总结 | Cell 32-36 | 5 min |
| 88-95 min | 练习：计算 DPO 奖励 | Cell 37-39 | 7 min |

---

## 课前准备

- [ ] 确认 `torch`, `transformers`, `numpy`, `matplotlib` 已安装
- [ ] 确认 `Qwen/Qwen2.5-0.5B-Instruct` 模型已下载至 `models/` 目录
- [ ] 可选：安装 `trl`, `datasets`（用于 Part 6 生产级演示）
- [ ] GPU 环境优先；CPU 可跑但训练部分耗时较长（15-30 分钟）
- [ ] 提前运行一遍 notebook 确认输出正常，特别是训练曲线和生成对比
- [ ] 准备白板或投影——数学推导部分需要手写辅助讲解


---

## 第一段：开场 + 对齐动机 + RLHF vs DPO（Cell 0-3）

📍 浏览 Cell 0（标题与导读）、Cell 1（学习路线）、Cell 2（对齐动机）、Cell 3（RLHF vs DPO 对比）

⏱ 时间分配：8 分钟

🎯 本段目标
- 建立学习动机：SFT 之后为什么还需要对齐？
- 通过「啰嗦 vs 简洁」的例子让学生直观理解对齐的价值
- 对比 RLHF 和 DPO 的流程差异，理解 DPO 的简洁优势

🗣 讲课话术

> 大家好！上一章我们做了 SFT 指令微调，模型学会了「能回答问题」。但是，能回答不等于回答得好。来看 Cell 2 的例子。
>
> 你问模型「水的沸点是多少？」，一个没对齐的模型可能回你一大段话——什么克劳修斯-克拉佩龙方程都搬出来了。而你只想听一句：「100°C」。这就是对齐要解决的问题：**不是让模型知道更多，而是让它学会用户想要的回答方式**。
>
> 对齐有三个维度：有帮助、诚实、无害。今天我们聚焦「简洁风格」这个任务——它是最直观可量化的：回复变短了就是对齐成功了。
>
> 来看 Cell 3 的流程对比。传统 RLHF 需要 4 步——先 SFT，再收集偏好数据，然后训练奖励模型，最后用 PPO 优化策略。而 DPO 只需要 2 步——SFT 加直接偏好优化，**把奖励模型直接消化进了损失函数里**。这是 2023 年 Rafailov 等人的一个非常优雅的工作。

👀 输出要点
- Cell 0 标题：任务是训练模型学会「简洁回复」风格
- Cell 2 的对比例子：啰嗦版 vs 简洁版
- Cell 3 的表格：RLHF = 复杂（4步），DPO = 简单（2步）

❓ 预判问题

Q: DPO 既然更简单，为什么还有人用 RLHF？
A: DPO 的前提是你有高质量的离线偏好数据。RLHF 可以通过在线采样不断更新数据分布，适合数据持续迭代的场景。工业级如 ChatGPT 早期用的就是 RLHF，但现在越来越多团队（包括 Llama 3）也开始用 DPO 或其变体。

Q: 对齐和 SFT 有什么区别？
A: SFT 教模型「能回答」——给它指令-回复对让它学会格式。对齐教模型「怎么回答更好」——在多个候选回答中学会偏好。SFT 是必要条件，对齐是锦上添花。

➡️ 转场

> 好，动机清楚了。我们先把环境跑起来，然后看看偏好数据长什么样。


---

## 第二段：环境准备（Cell 4-6）

📍 运行 Cell 5（导入依赖 + 设备检测）、Cell 6（加载 Transformers）

⏱ 时间分配：5 分钟

🎯 本段目标
- 确认运行环境就绪
- 检测 GPU/CPU 设备
- 加载基础依赖

🗣 讲课话术

> 来看 Cell 4 的依赖列表——核心就是 torch 和 transformers，加上 numpy 和 matplotlib 做可视化。trl 和 datasets 是可选的，后面 Part 6 会用到。
>
> 运行 Cell 5。（运行 Cell 5）看输出——设备是 CUDA，GPU 是 RTX 5080。如果你用 CPU 也没关系，所有代码都能跑，就是训练那一步会慢一些。
>
> 运行 Cell 6。（运行 Cell 6）输出「Transformers loaded!」就行了。

👀 输出要点
- Cell 5：显示 `Using device: cuda` 和 GPU 型号
- Cell 6：输出 `Transformers loaded!`

❓ 预判问题

Q: CPU 跑得动吗？
A: 跑得动。notebook 会自动调整超参——CPU 用更小的 batch size 和更少的 epoch。训练可能需要 15-30 分钟，但效果一样能看到。

➡️ 转场

> 环境 OK。下面进入正题——先看偏好数据是什么结构。


---

## 第三段：偏好数据构造与分析（Cell 7-8）

📍 浏览 Cell 7（Markdown 讲解）、运行 Cell 8（偏好数据定义 + 统计可视化）

⏱ 时间分配：10 分钟

🎯 本段目标
- 理解偏好数据的三元组结构：(prompt, chosen, rejected)
- 通过统计数据建立直觉：简洁 vs 冗长的量化差异
- 理解为什么选「简洁风格」作为教学任务

🗣 讲课话术

> DPO 的核心输入就是偏好数据。每一条都是三元组——一个问题，一个好回答（chosen），一个差回答（rejected）。
>
> 来看 Cell 8 的数据。比如问「中国的首都是哪里？」，chosen 是「北京。」两个字，rejected 是一大段介绍。我们一共有 **30 条**偏好对。
>
> 运行 Cell 8。（运行 Cell 8）看统计结果——简洁回复平均 **15.1 字符**，冗长回复平均 **91.4 字符**，长度比是 **6.1 倍**！简洁回复最短只有 2 个字符，最长 46 个字符；冗长回复最短 46 个，最长 142 个。
>
> 看右边的直方图——两类回复的长度分布几乎没有重叠。这就是 DPO 要学的信号：「短的好，长的不好」。
>
> 思考题：为什么我选「简洁 vs 冗长」而不是「正确 vs 错误」来教 DPO？因为风格偏好是 DPO 最擅长的场景——效果可量化（看长度变化），训练前后差异显著。而「正确性」需要模型本身有足够知识，小模型很难展示。

👀 输出要点
- 偏好数据集：30 条
- 简洁回复平均长度：15.1 字符
- 冗长回复平均长度：91.4 字符
- 长度比：6.1x
- 简洁回复长度范围：[2, 46] 字符
- 冗长回复长度范围：[46, 142] 字符
- 直方图：两类分布清晰分离

❓ 预判问题

Q: 30 条数据够吗？真实场景要多少？
A: 教学演示足够了。生产环境通常需要数千到数万条偏好对。比如 Anthropic 的 HH-RLHF 数据集有 16 万条。质量比数量更重要——噪声大的偏好数据反而有害。

Q: 偏好数据怎么收集？
A: 主要两种方式：(1) 人工标注——给标注员看两个回答，选更好的那个；(2) AI 辅助——用更强的模型（如 GPT-4）当裁判打分。实际中常用 AI 初筛 + 人工抽检。

➡️ 转场

> 数据结构清楚了。下面是本章最核心的部分——DPO 的数学原理。别怕，公式不多，而且直觉很自然。


---

## 第四段：DPO 数学原理（Cell 9-14）

📍 浏览 Cell 9（DPO 公式概览）、Cell 10（完整推导）、Cell 11（术语锚点）、Cell 12（beta 超参数与变体）、运行 Cell 13（损失函数代码）、运行 Cell 14（损失可视化）

⏱ 时间分配：20 分钟（推导 10 分钟 + beta 分析 5 分钟 + 可视化 5 分钟）

🎯 本段目标
- 理解 DPO 损失函数的推导过程：从 RLHF 目标到闭式解到 DPO loss
- 建立「隐式奖励」的直觉：奖励 = beta x log(策略/参考)
- 理解 beta 超参数的作用和调参直觉
- 通过可视化加深对损失函数形状的理解

🗣 讲课话术

> 这部分是 DPO 最精华的地方，我们慢慢来。
>
> 先看 Cell 9 的 RLHF 目标函数。RLHF 想做的事很直观：让策略模型得到高奖励，同时不能偏离参考模型太远。后面那个 KL 散度项就是「不要跑太偏」的约束，beta 控制约束有多强。
>
> 关键洞见来了——这个带 KL 约束的优化问题，居然有**精确的闭式解**。Cell 10 给了完整推导，但核心就一步：把最优策略的表达式反过来解，你会发现**奖励可以用策略和参考模型的 log 比值来表达**。
>
> 写在白板上：r(x,y) = beta x log(pi_theta(y|x) / pi_ref(y|x)) + 常数项。这就是**隐式奖励**——我们不需要单独训练一个奖励模型了！奖励就藏在策略模型和参考模型的概率比里。
>
> 把这个代入 Bradley-Terry 偏好模型（就是 P(A 比 B 好) = sigmoid(奖励差)），常数项抵消，就得到 DPO 的损失函数——Cell 9 底部那个公式。
>
> 来看 Cell 11 的术语锚点表，把中英文对应记住。特别注意：「log ratio」是 log(pi_theta/pi_ref)，「log ratio diff」是 chosen 的 log ratio 减去 rejected 的 log ratio。
>
> Cell 12 讲 beta 调参。看表格——beta 太小（<0.05），模型变化太大，容易模式坍缩；beta 太大（>1.0），几乎不变，白训。**0.1 到 0.5 是甜区**。最终我们 CGT_04 用的是 **β=0.5**——在 2.73M 小模型 + 150 条偏好对的演示场景里，β=0.1 会让 policy 漂离 ref 太远导致生成乱码（详见 CGT_04 教案的「为什么 β 不是越小越好」段）；β=0.5 加上 max_steps=10 的早停，policy 几乎没动但准确率从 50% 升到 100%。
>
> 数值直觉：beta=0.1 时，log-ratio 差异为 1.0 对应 sigma(0.1 x 1.0) = 0.525（微弱偏好）；差异为 10.0 对应 sigma(1.0) = 0.731（明显偏好）。
>
> 运行 Cell 13。（运行 Cell 13）DPO loss 函数定义好了。核心就 4 行代码——算 chosen log ratio、rejected log ratio、取差值、过 logsigmoid。
>
> 运行 Cell 14。（运行 Cell 14）看这三张图。左图：不同 beta 下的损失曲线。beta 越大，曲线越陡——意味着对 log ratio 差异更敏感。中图：梯度曲线。右图：损失的概率解释——sigma(beta x Delta) 就是模型对 chosen 的偏好概率。

👀 输出要点
- Cell 13：输出 `DPO loss function defined.`
- Cell 14 三张图：
  - 左图：beta=0.05/0.1/0.2/0.5 四条损失曲线
  - 中图：梯度曲线
  - 右图：偏好概率 sigma(beta x Delta)

❓ 预判问题

Q: 为什么 DPO 不需要奖励模型？
A: 因为 RLHF 的 KL 约束优化问题恰好有闭式解，从中可以把奖励表达为策略和参考模型的 log 比值。这样就把奖励模型「隐式化」了——不需要单独训练，只需要两个语言模型的概率比。

Q: beta 怎么选？
A: 经验法则是从 0.1 开始。如果对齐效果不够强就调小，如果模型生成质量下降（比如开始胡说）就调大。Cell 12 的表格给了直觉：beta=0.1 时，log ratio 差 10.0 对应偏好概率 0.731。

Q: 推导里的 Z(x) 是什么？
A: 归一化常数（partition function），确保最优策略是合法概率分布。在 DPO 中它被巧妙消掉了——因为 chosen 和 rejected 共用同一个 prompt x，Z(x) 在做差时抵消。这是 DPO 能 work 的关键数学技巧。

➡️ 转场

> 数学讲完了，我们休息一下再动手训练。


---

## 休息 + 回顾（第 43-48 分钟）

⏱ 时间分配：5 分钟

**三句话回顾前半段：**

1. 对齐的目标是让模型不只「能回答」，而是「回答得好」。我们用「简洁 vs 冗长」作为可量化的风格任务，30 条偏好对，简洁平均 15.1 字符、冗长 91.4 字符，6.1 倍差距。
2. DPO 的核心洞见：RLHF 的带 KL 约束优化有闭式解，奖励可以用 log(pi_theta/pi_ref) 表达，于是不需要单独训练奖励模型。
3. DPO 损失函数 = -log sigma(beta x Delta)，其中 Delta 是 chosen 与 rejected 的 log ratio 差。beta 控制对齐强度，0.1-0.5 是常用范围。

**下一段预告：** 我们要加载 Qwen2.5-0.5B 模型，看看训练前它回答问题有多啰嗦，然后用 DPO 训练让它变简洁。


---

## 第五段：模型加载与配置（Cell 15-18）

📍 浏览 Cell 15（Markdown 说明）、运行 Cell 16（训练配置）、运行 Cell 17（加载模型）、运行 Cell 18（基线测试生成）

⏱ 时间分配：10 分钟

🎯 本段目标
- 理解 DPO 需要两个模型：policy（可训练）+ reference（冻结）
- 了解「只训练最后几层」的参数高效策略
- 看到训练前模型的基线表现：啰嗦的回复风格

🗣 讲课话术

> Cell 15 说明了 DPO 的双模型架构。policy_model 是我们要训练的，reference_model 是冻结不动的参考线。为什么需要参考模型？因为 DPO 的 loss 里有 log(pi_theta/pi_ref)——没有参考模型就算不出来。
>
> 运行 Cell 16。（运行 Cell 16）看配置——用的是 Qwen2.5-0.5B-Instruct，5 亿参数的小模型。GPU 配置：beta=0.1，学习率 5e-6，batch size 4，训练 3 个 epoch。只训练最后 4 层加 lm_head。
>
> 运行 Cell 17。（运行 Cell 17）看输出——总参数 **4.94 亿**，但可训练参数只有 **5960 万**，占 **12.1%**。用的是 bfloat16 精度。为什么只训最后几层？因为风格偏好主要体现在输出层附近，不需要改底层表示。而且省显存。
>
> 运行 Cell 18。（运行 Cell 18）这是训练前的基线生成。看第一个问题「中国的首都是哪里？」——模型回了 **97 个字**，扯了一堆省级行政区。我们期望 DPO 训练后这个数字能显著下降。

👀 输出要点
- Cell 16：DPO 训练配置表（模型名、设备、beta=0.1、lr=5e-6、batch=4、epochs=3）
- Cell 17：总参数 494M，可训练 59.6M (12.1%)，dtype bfloat16
- Cell 18：基线生成——「中国的首都是哪里？」回复 97 字

❓ 预判问题

Q: 为什么不训练全部参数？
A: 两个原因：(1) 省显存——0.5B 模型全量训练需要约 4GB，冻结大部分层后只需要约 1GB 梯度内存；(2) 风格对齐不需要改变底层知识表示，只需要调整输出分布。实际上只训练最后几层在 DPO 中是常见做法。

Q: reference_model 为什么要冻结？
A: 它是 KL 约束的锚点。如果 reference 也在变，你的「不要偏离太远」约束就失去了意义——好比用一把会伸缩的尺子量距离。

➡️ 转场

> 基线看到了——模型确实很啰嗦。下面开始 DPO 训练，看看能不能把它教简洁。


---

## 第六段：DPO 训练实战（Cell 19-26）

📍 浏览 Cell 19（训练说明）、运行 Cell 20（DPODataset）、浏览 Cell 21（compute_log_probs）、浏览 Cell 22（log-ratio 差解释）、运行 Cell 23（DPOTrainer 类）、运行 Cell 24（训练主循环）、运行 Cell 25（保存模型）、运行 Cell 26（训练曲线可视化）

⏱ 时间分配：15 分钟（数据准备 3 分钟 + 训练 7 分钟 + 可视化 5 分钟）

🎯 本段目标
- 理解 DPO 训练的完整流程：数据准备 -> log-prob 计算 -> loss -> 参数更新
- 理解「只统计回复部分 log-prob」的技巧——避免 prompt 干扰
- 观察训练过程中三个关键指标的变化：Loss、Reward Margin、Accuracy
- 通过训练曲线判断训练是否正常

🗣 讲课话术

> 进入 Part 4——训练实战。先看 Cell 19 的训练步骤总结：计算 chosen 和 rejected 的 log-prob，算 log-ratio 差，过 DPO loss，更新参数。
>
> 运行 Cell 20。（运行 Cell 20）Dataset 准备好了，30 条数据。注意这里有个关键细节——我们记录了 prompt_length，因为**算 log-prob 时只统计回复部分**。为什么？如果把 prompt 也算进去，那 log-prob 的差异会被 prompt 的概率「淹没」——毕竟 chosen 和 rejected 共享同一个 prompt，它的概率贡献是噪声。
>
> Cell 21 定义了 compute_log_probs 函数。核心思路：先算整个序列每个 token 的 log-prob，然后用 prompt_length 做 mask，只保留回复部分。reduction="mean" 对 token 数取平均，避免长回复天然获得更低的 log-prob。
>
> Cell 22 再次强调 log-ratio 差的含义——当 Delta > 0 时，模型比参考模型更偏好 chosen。DPO 训练的目标就是把 Delta 推向正方向。
>
> 运行 Cell 23 定义 DPOTrainer。运行 Cell 24 开始训练。（运行 Cell 24）
>
> 看训练日志——Step 5 时 Loss 是 0.6929（接近 -log(0.5) = 0.693，说明模型还没学到偏好），Margin 是 0.0045（接近 0，chosen 和 rejected 还没分开），Accuracy 75%。随着训练推进，Loss 应该下降，Margin 应该变大，Accuracy 应该提高。
>
> GPU 上训练 30 条数据 3 个 epoch 大约 1-3 分钟。
>
> 运行 Cell 25 保存模型。运行 Cell 26 看训练曲线。（运行 Cell 26）三张图——Loss 在下降、Reward Margin 在增大、Preference Accuracy 在提高。这说明训练正常，模型在学习偏好 chosen 而非 rejected。

👀 输出要点
- Cell 20：Dataset: 30 examples
- Cell 24 训练日志：
  - Step 5: Loss 0.6929, Margin 0.0045, Acc 75%
  - 随后 Loss 逐步下降
- Cell 25：模型保存到 `models/ch10_dpo_custom`
- Cell 26 三张图：Loss 下降、Reward Margin 上升、Accuracy 提高

❓ 预判问题

Q: 初始 Loss 为什么接近 0.693？
A: 因为 -log(sigma(0)) = -log(0.5) = 0.693。训练开始时 policy 和 reference 完全一样，log ratio 差 Delta=0，所以 loss 就是 log(2)。这是个好的 sanity check——如果初始 loss 不在 0.69 附近，说明代码有 bug。

Q: 为什么只算回复部分的 log-prob？
A: 因为 prompt 对 chosen 和 rejected 是相同的。如果算整个序列的 log-prob，prompt 部分的概率差异只是随机噪声，会稀释回复部分的信号。这是 DPO 实现中一个容易踩的坑。

Q: Reward Margin 负值说明什么？
A: 说明模型目前更偏好 rejected 而非 chosen。训练的目标就是把 margin 推到正值。如果训练到后期 margin 还是负的，需要检查数据质量或调大学习率。

➡️ 转场

> 训练完成了。关键问题来了——模型真的变简洁了吗？我们做个 AB 对比。


---

## 第七段：效果评估与隐式奖励（Cell 27-31）

📍 浏览 Cell 27（评估说明）、运行 Cell 28（Reference vs DPO 对比生成）、运行 Cell 29（量化统计）、运行 Cell 30（偏好准确率）、运行 Cell 31（隐式奖励可视化）

⏱ 时间分配：10 分钟

🎯 本段目标
- 直观看到 DPO 训练前后回复风格的变化
- 通过量化指标评估对齐效果：长度变化、偏好准确率、隐式奖励
- 区分训练集内（in-distribution）和训练集外（OOD）的泛化表现
- 理解隐式奖励 r_theta(x,y) = beta x log(pi_theta/pi_ref) 的实际意义

🗣 讲课话术

> Part 5——验收时间。Cell 28 同时测试训练集内的 3 个问题和训练集外的 5 个问题。
>
> 运行 Cell 28。（运行 Cell 28）看对比——「中国的首都是哪里？」Reference 模型回了 97 个字，DPO 模型回了 65 个字，**缩短了 32 个字**。不过也有些问题反而变长了一点，比如「什么是机器学习？」从 177 字变成 184 字。
>
> 运行 Cell 29 看量化统计。（运行 Cell 29）Reference 平均 **155.0 字符**，DPO 平均 **151.2 字符**，长度减少 **2.4%**。减少幅度不大——这是因为我们只训练了最后 4 层，而且只有 30 条数据 3 个 epoch。生产中用全量参数 + 更多数据效果会显著得多。
>
> 运行 Cell 30 看偏好准确率。（运行 Cell 30）Reference 的偏好准确率只有 **16.7%**——也就是说原始模型大部分情况下更偏好冗长回复！DPO 训练后提升到 **23.3%**，提升了 **6.7 个百分点**。虽然绝对值不高，但方向是对的——记住我们只用了 30 条数据训了几分钟。
>
> 运行 Cell 31——这是最直观的图。（运行 Cell 31）隐式奖励柱状图显示：**30/30 条数据的 chosen 奖励都高于 rejected 奖励**，100% 胜率！Chosen 平均隐式奖励 **0.0198**，Rejected 平均 **-0.0018**。模型确实学会了「简洁更好」这个偏好。

👀 输出要点
- Cell 28：逐题对比，「中国的首都是哪里？」从 97 字缩短到 65 字
- Cell 29：Reference 平均 155.0 字符，DPO 平均 151.2 字符，减少 2.4%
- Cell 30：偏好准确率从 16.7% 提升到 23.3%（+6.7%）
- Cell 31：Chosen reward > Rejected reward: 30/30 (100%)
  - Chosen 平均隐式奖励: 0.0198
  - Rejected 平均隐式奖励: -0.0018

❓ 预判问题

Q: 长度只减少 2.4%，算成功吗？
A: 对于 30 条数据 + 只训练 12% 的参数 + 3 个 epoch 来说，方向正确比幅度重要。隐式奖励可视化显示 100% 的偏好方向都对了。实际部署中，用更多数据（几千条）和更多训练步数，效果会显著得多。

Q: 偏好准确率 23.3% 不是还很低吗？
A: 偏好准确率衡量的是模型给 chosen 的 log-prob 是否高于 rejected。因为 chosen 是极短的回答（平均 15 字符），而 rejected 更长，在 mean reduction 下短回复的 per-token log-prob 未必更高。这个指标要结合隐式奖励一起看——Cell 31 的 100% 胜率才是真正反映 DPO 效果的指标。

Q: 隐式奖励是怎么算的？
A: r_theta(x,y) = beta x log(pi_theta(y|x) / pi_ref(y|x))。就是策略模型和参考模型的概率比，乘以 beta。如果策略模型比参考模型更可能生成这个回答，隐式奖励为正；反之为负。

➡️ 转场

> 我们的简易 DPO 训练效果验证完了。最后快速看一下生产级实现和总结。


---

## 第八段：生产级实现 + 总结（Cell 32-36）

📍 浏览 Cell 32（TRL 说明）、运行 Cell 33（TRL DPOTrainer 演示）、浏览 Cell 34（总结图谱）、浏览 Cell 35（下一步）、浏览 Cell 36（Extra 练习建议）

⏱ 时间分配：5 分钟

🎯 本段目标
- 了解 TRL 库的 DPOTrainer 作为生产级替代方案
- 串联全章知识点

🗣 讲课话术

> Cell 33 展示了 TRL 库的 DPOTrainer——这是 Hugging Face 官方的生产级实现。如果你的环境装了 trl 和 datasets，运行看看。核心区别是什么？我们手写了 200 多行代码，TRL 只需要几行配置就搞定了——它帮你处理了数据预处理、梯度累积、日志、checkpoint 等所有工程细节。
>
> 看 Cell 33 的输出——TRL 训练的 loss 是 0.545，rewards/chosen=0.1464，rewards/rejected=-0.1802，margin=0.3266，准确率 100%。效果比我们手写的更好，因为 TRL 内部有更多优化。
>
> Cell 34 的核心概念图谱把全章内容串起来了——偏好数据进入 DPO Loss，更新策略模型，通过隐式奖励评估效果。Cell 35 指向下一章 KV Cache 推理优化和 Bonus A 的 RLHF 全景。

👀 输出要点
- Cell 33 TRL 训练指标：loss=0.545, margin=0.3266, accuracy=100%
- Cell 34：全章知识图谱

❓ 预判问题

Q: 生产中用手写 DPO 还是 TRL？
A: 生产用 TRL（或类似框架如 OpenRLHF、DeepSpeed-Chat）。手写是为了理解原理。就像你需要理解反向传播，但不会在生产中手写 autograd。

➡️ 转场

> 最后一个练习——动手写 DPO 的核心计算。


---

## 第九段：练习——计算 DPO 奖励（Cell 37-39）

📍 浏览 Cell 37（练习说明）、Cell 38（练习代码，有 TODO）、运行 Cell 39（参考答案与验证）

⏱ 时间分配：7 分钟

🎯 本段目标
- 学生独立实现 DPO 隐式奖励的核心计算
- 验证理解是否到位

🗣 讲课话术

> Cell 38 是一个填空练习——实现 compute_dpo_reward 函数。需要你写三步：
> 1. 计算 chosen 的 log-ratio：policy_chosen_logp - ref_chosen_logp
> 2. 计算 rejected 的 log-ratio：policy_rejected_logp - ref_rejected_logp
> 3. 算 reward_margin（差值）和 loss（过 logsigmoid）
>
> 给你 2 分钟自己试。

### 提示节奏

**0-2 分钟：** 独立尝试。提示：回忆 Cell 9 的公式和 Cell 13 的代码。

**2 分钟第一个提示：**

> log-ratio 就是策略和参考的 log-prob 做减法。chosen 算一次，rejected 算一次。

**4 分钟给出关键代码：**
```python
chosen_log_ratio = policy_chosen_logp - ref_chosen_logp
rejected_log_ratio = policy_rejected_logp - ref_rejected_logp
reward_margin = chosen_log_ratio - rejected_log_ratio
loss = -F.logsigmoid(beta * reward_margin)
```

### 常见错误
1. **忘记乘 beta**：loss = -F.logsigmoid(reward_margin) -- 缺少 beta 缩放
2. **log-ratio 方向反了**：写成 ref - policy 而非 policy - ref
3. **用 sigmoid 而非 logsigmoid**：数值稳定性差，大负数时会下溢

### 验证标准
运行 Cell 39 的验证代码，应输出：
- `✅ 正确！你已掌握 DPO 的核心计算`
- chosen log-ratio: 0.50
- rejected log-ratio: -0.50
- reward margin (Delta): 1.00
- DPO loss: 0.6444
- `Delta > 0 -- 模型比 ref 更偏好 chosen`


---

## 附录

### 时间表汇总

| 时间段 | 内容 | Cell 范围 | 时长 |
|:---|:---|:---|:---|
| 0-8 min | 开场 + 对齐动机 + RLHF vs DPO | Cell 0-3 | 8 min |
| 8-13 min | 环境准备 | Cell 4-6 | 5 min |
| 13-23 min | 偏好数据 | Cell 7-8 | 10 min |
| 23-43 min | DPO 数学原理 | Cell 9-14 | 20 min |
| 43-48 min | 休息 | — | 5 min |
| 48-58 min | 模型加载 | Cell 15-18 | 10 min |
| 58-73 min | DPO 训练 | Cell 19-26 | 15 min |
| 73-83 min | 效果评估 | Cell 27-31 | 10 min |
| 83-88 min | 生产级 + 总结 | Cell 32-36 | 5 min |
| 88-95 min | 练习 | Cell 37-39 | 7 min |

### 关键数据速查

| 数据项 | 值 |
|:---|:---|
| 偏好数据量 | 30 条 |
| 简洁回复平均长度 | 15.1 字符 |
| 冗长回复平均长度 | 91.4 字符 |
| 长度比 | 6.1x |
| 模型 | Qwen2.5-0.5B-Instruct (494M 参数) |
| 可训练参数 | 59.6M (12.1%) |
| beta | 0.1 |
| 学习率 | 5e-6 |
| 训练 epochs | 3 |
| 初始 Loss | ~0.693 (= -log(0.5)) |
| 训练后长度减少 | 2.4% |
| 偏好准确率 | 16.7% -> 23.3% |
| 隐式奖励胜率 | 30/30 (100%) |
| Chosen 平均隐式奖励 | 0.0198 |
| Rejected 平均隐式奖励 | -0.0018 |
| TRL loss | 0.545, margin=0.3266, accuracy=100% |

### 应急预案

| 场景 | 应对 |
|:---|:---|
| GPU 显存不足 | 减小 MAX_SEQ_LEN 至 128，DPO_BATCH_SIZE 至 1，TRAINABLE_LAYERS 至 2 |
| 训练时间过长（CPU） | 减少 DPO_EPOCHS 至 1，跳过 Cell 33 TRL 演示 |
| 模型下载失败 | 确认 models/ 目录有预下载模型，或切换为 distilgpt2 演示核心概念 |
| TRL 未安装 | 跳过 Cell 33，手写实现已覆盖全部核心内容 |
| 训练 Loss 不下降 | 检查学习率（试 1e-5），检查 prompt_length 是否正确，确认 reference 模型已冻结 |
| 学生 GPU 环境不一致 | Cell 16 自动区分 cuda/cpu 配置，提醒学生信任默认值 |
| 数学推导时间不够 | 重点讲 Cell 9 的三个公式 + Cell 14 的可视化，Cell 10 完整推导留作课后阅读 |
